# 05 – Post-Processing & Profili di Rischio

**Obiettivo**: applicare un livello di *business rules* sui risultati ML di `04_anomaly_detection.ipynb`  
per ottenere:
1. Un **livello di rischio finale** (CRITICO / ALTO / MEDIO / BASSO) per ogni rotta
2. Un **confidence score** combinato (ML + regole)
3. **Profili di rischio** per rotta, paese, zona
4. Un dataset finale pronto per il confronto con il pipeline Multi-Agent

**Input**: `anomaly_results.csv` · `features_with_baseline.csv` · `baseline_stats.json`  
**Output**: `final_report.csv` · `risk_profiles.json` · `country_risk.csv`

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

BASE = '../data/processed'
print('✓ Librerie caricate')

✓ Librerie caricate


## 1. Caricamento dati

In [2]:
# Risultati anomaly detection (ML)
ar = pd.read_csv(f'{BASE}/anomaly_results.csv')

# Feature engineering completo + baseline flags
fb = pd.read_csv(f'{BASE}/features_with_baseline.csv')

# Statistiche baseline
bs_raw = json.load(open(f'{BASE}/baseline_stats.json'))
bs = bs_raw['stats']

print(f'anomaly_results    : {ar.shape[0]:>4} rotte × {ar.shape[1]} colonne')
print(f'features_baseline  : {fb.shape[0]:>4} rotte × {fb.shape[1]} colonne')
print(f'baseline_stats     : {len(bs)} feature monitorate')

anomaly_results    :  567 rotte × 16 colonne
features_baseline  :  567 rotte × 87 colonne
baseline_stats     : 11 feature monitorate


## 2. Merge ML + Feature Set

In [3]:
# Evita duplicati: teniamo le colonne già presenti in fb e solo i punteggi ML da ar
ar_cols = ['ROTTA', 'anomaly_score', 'anomaly_label', 'rank']
df = fb.merge(ar[ar_cols], on='ROTTA', how='left')

# Fill: rotte senza score ML → NORMALE con score 0
df['anomaly_score']  = df['anomaly_score'].fillna(0.0)
df['anomaly_label']  = df['anomaly_label'].fillna('NORMALE')
df['rank']           = df['rank'].fillna(999).astype(int)

print(f'Dataset unificato: {df.shape[0]} rotte × {df.shape[1]} colonne')
print(f'Distribuzione ML label: {dict(df["anomaly_label"].value_counts())}')

Dataset unificato: 567 rotte × 90 colonne
Distribuzione ML label: {'NORMALE': np.int64(510), 'MEDIA': np.int64(40), 'ALTA': np.int64(17)}


## 3. Business Rules

Cinque regole indipendenti che catturano pattern operativi rilevanti per la sicurezza aeroportuale:

| Regola | Condizione | Logica |
|--------|-----------|--------|
| **BR1 Volume** | `tot_allarmi_log > μ + 2σ` | Rotte con volume allarmi > soglia statistica |
| **BR2 INTERPOL** | `pct_interpol > 0.25` | >25% allarmi da banche dati INTERPOL |
| **BR3 Esiti** | `score_rischio_esiti ≥ 0.4` | Alta incidenza di respinti / fermati |
| **BR4 Multi-segnale** | `n_anomalie ≥ 3` | ≥3 feature statisticamente anomale (03_baseline) |
| **BR5 SDI/NSIS** | `pct_sdi > 0.20 ∨ pct_nsis > 0.20` | Significativa presenza banche dati nazionali |


In [4]:
vol_mean = bs['tot_allarmi_log']['mean']
vol_std  = bs['tot_allarmi_log']['std']
VOL_THRESHOLD = vol_mean + 2.0 * vol_std

df['br_volume']      = (df['tot_allarmi_log'] > VOL_THRESHOLD).astype(int)
df['br_interpol']    = (df['pct_interpol'] > 0.25).astype(int)
df['br_esiti']       = (df['score_rischio_esiti'] >= 0.40).astype(int)
df['br_multisignal'] = (df['n_anomalie'] >= 3).astype(int)
df['br_sdi_nsis']    = ((df['pct_sdi'] > 0.20) | (df['pct_nsis'] > 0.20)).astype(int)

BR_COLS = ['br_volume','br_interpol','br_esiti','br_multisignal','br_sdi_nsis']
df['br_score'] = df[BR_COLS].sum(axis=1)

print(f'Soglia volume (μ+2σ): {VOL_THRESHOLD:.3f}')
print()
for col in BR_COLS:
    n = df[col].sum()
    pct = 100*n/len(df)
    print(f'  {col:<20}: {n:>3} rotte  ({pct:.1f}%)')
print()
print('BR score distribution:')
print(df['br_score'].value_counts().sort_index())

Soglia volume (μ+2σ): 9.170

  br_volume           :   4 rotte  (0.7%)
  br_interpol         :  86 rotte  (15.2%)
  br_esiti            :  55 rotte  (9.7%)
  br_multisignal      :  11 rotte  (1.9%)
  br_sdi_nsis         : 235 rotte  (41.4%)

BR score distribution:
br_score
0    240
1    268
2     55
3      3
4      1
Name: count, dtype: int64


## 4. Confidence Score

Il **confidence score** combina il segnale ML con le business rules:

```
confidence = 0.60 × anomaly_score + 0.40 × (br_score / 5)
```

- Il punteggio ML ha peso maggiore (modelli validati statisticamente)
- Le business rules agiscono come amplificatore / filtro operativo


In [5]:
df['confidence'] = (
    0.60 * df['anomaly_score'] +
    0.40 * (df['br_score'] / 5.0)
).round(4)

print('Confidence score — statistiche descrittive:')
print(df['confidence'].describe().round(4))
print()
print(f'Rotte con confidence > 0.50: {(df["confidence"] > 0.50).sum()}')
print(f'Rotte con confidence > 0.40: {(df["confidence"] > 0.40).sum()}')
print(f'Rotte con confidence > 0.30: {(df["confidence"] > 0.30).sum()}')

Confidence score — statistiche descrittive:
count   567.0000
mean      0.1410
std       0.1014
min       0.0005
25%       0.0422
50%       0.1528
75%       0.2086
max       0.5945
Name: confidence, dtype: float64

Rotte con confidence > 0.50: 2
Rotte con confidence > 0.40: 6
Rotte con confidence > 0.30: 33


## 5. Classificazione Finale del Rischio

La matrice ML × Business Rules produce quattro livelli:

| ML Label | BR Score | Risk Level |
|----------|----------|------------|
| ALTA     | ≥ 2      | **CRITICO** |
| ALTA     | < 2      | **ALTO** |
| MEDIA    | ≥ 2      | **ALTO** |
| MEDIA    | ≥ 1      | **MEDIO** |
| MEDIA    | 0        | **MEDIO** |
| NORMALE  | ≥ 3      | **MEDIO** |
| NORMALE  | < 3      | **BASSO** |


In [6]:
def classify_risk(row):
    ml = row['anomaly_label']
    br = row['br_score']
    if   ml == 'ALTA'    and br >= 2: return 'CRITICO'
    elif ml == 'ALTA':               return 'ALTO'
    elif ml == 'MEDIA'   and br >= 2: return 'ALTO'
    elif ml == 'MEDIA':              return 'MEDIO'
    elif ml == 'NORMALE' and br >= 3: return 'MEDIO'
    else:                            return 'BASSO'

df['risk_level'] = df.apply(classify_risk, axis=1)

# Ordine visivo
RISK_ORDER = {'CRITICO':4,'ALTO':3,'MEDIO':2,'BASSO':1}
df['risk_num'] = df['risk_level'].map(RISK_ORDER)

print('=== DISTRIBUZIONE RISK LEVEL ===')
vc = df['risk_level'].value_counts()
for level in ['CRITICO','ALTO','MEDIO','BASSO']:
    n = vc.get(level, 0)
    bar = '█' * n
    print(f'  {level:<8}: {n:>3}  {bar}')

=== DISTRIBUZIONE RISK LEVEL ===
  CRITICO :   6  ██████
  ALTO    :  16  ████████████████
  MEDIO   :  35  ███████████████████████████████████
  BASSO   : 510  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████


## 6. Top Rotte per Rischio

In [7]:
DISPLAY_COLS = [
    'ROTTA','PAESE_PART','ZONA','risk_level','anomaly_label',
    'br_score','anomaly_score','confidence',
    'tot_allarmi_log','pct_interpol','score_rischio_esiti','n_anomalie'
]

top = df.nlargest(25, 'confidence')[DISPLAY_COLS].reset_index(drop=True)
top.index += 1
print('Top 25 rotte per confidence score:')
print(top.to_string())

Top 25 rotte per confidence score:
      ROTTA      PAESE_PART ZONA risk_level anomaly_label  br_score  anomaly_score  confidence  tot_allarmi_log  pct_interpol  score_rischio_esiti  n_anomalie
1   ALG-MXP         Algeria  2.0    CRITICO          ALTA         3         0.5909      0.5945           5.1874        0.2500               0.4000           3
2   JED-VCE  Arabia Saudita  4.0       ALTO         MEDIA         4         0.3376      0.5226           5.3613        0.5000               0.4000           3
3   RUH-VCE  Arabia Saudita  4.0    CRITICO          ALTA         3         0.3985      0.4791           1.6094        0.0000               0.6000           4
4   RAK-CIA         Marocco  5.0    CRITICO          ALTA         2         0.5253      0.4752           3.9512        0.1000               0.4800           2
5   EVN-VCE         Armenia  4.0       ALTO         MEDIA         3         0.3381      0.4429           4.7449        0.0000               0.6000           3
6   CMN-BLQ

## 7. Filtro per Soglia di Confidenza

Si applicano due filtri operativi:
- Livello di rischio ≥ MEDIO (esclude BASSO)
- Confidence ≥ 0.20 (evita falsi positivi a basso segnale)


In [8]:
CONF_THRESHOLD = 0.20
RISK_THRESHOLD = ['CRITICO','ALTO','MEDIO']

df_alert = df[
    (df['risk_level'].isin(RISK_THRESHOLD)) &
    (df['confidence'] >= CONF_THRESHOLD)
].copy().sort_values(['risk_num','confidence'], ascending=[False,False]).reset_index(drop=True)

print(f'Rotte in allerta (risk ≥ MEDIO, conf ≥ {CONF_THRESHOLD}): {len(df_alert)}')
print()
print('Distribuzione per risk level:')
print(df_alert['risk_level'].value_counts())
print()
print('Top 10:')
print(df_alert.head(10)[DISPLAY_COLS].to_string())

Rotte in allerta (risk ≥ MEDIO, conf ≥ 0.2): 52

Distribuzione per risk level:
risk_level
MEDIO      30
ALTO       16
CRITICO     6
Name: count, dtype: int64

Top 10:
     ROTTA      PAESE_PART ZONA risk_level anomaly_label  br_score  anomaly_score  confidence  tot_allarmi_log  pct_interpol  score_rischio_esiti  n_anomalie
0  ALG-MXP         Algeria  2.0    CRITICO          ALTA         3         0.5909      0.5945           5.1874        0.2500               0.4000           3
1  RUH-VCE  Arabia Saudita  4.0    CRITICO          ALTA         3         0.3985      0.4791           1.6094        0.0000               0.6000           4
2  RAK-CIA         Marocco  5.0    CRITICO          ALTA         2         0.5253      0.4752           3.9512        0.1000               0.4800           2
3  CMN-BLQ         Marocco  8.0    CRITICO          ALTA         2         0.4583      0.4350           6.8244        0.3043               0.4000           2
4  LHR-VCE     Regno Unito  4.0    CRITICO 

## 8. Analisi per Paese di Partenza

In [9]:
country_risk = (
    df[df['PAESE_PART'] != 'ND']
    .groupby('PAESE_PART')
    .agg(
        n_rotte        = ('ROTTA',         'count'),
        n_alert        = ('risk_level',     lambda x: (x.isin(['CRITICO','ALTO','MEDIO'])).sum()),
        max_risk_num   = ('risk_num',       'max'),
        max_confidence = ('confidence',     'max'),
        mean_confidence= ('confidence',     'mean'),
        tot_allarmi    = ('tot_allarmi_sum', 'sum'),
        mean_interpol  = ('pct_interpol',   'mean'),
        mean_esiti     = ('score_rischio_esiti','mean'),
    )
    .reset_index()
)
country_risk['max_risk_label'] = country_risk['max_risk_num'].map(
    {4:'CRITICO',3:'ALTO',2:'MEDIO',1:'BASSO'}
).fillna('BASSO')
country_risk['alert_rate'] = (
    country_risk['n_alert'] / country_risk['n_rotte']
).round(3)

top_countries = country_risk.nlargest(15,'max_confidence')[
    ['PAESE_PART','n_rotte','n_alert','alert_rate','max_risk_label','max_confidence','mean_interpol','mean_esiti']
].reset_index(drop=True)
top_countries.index += 1
print('Top 15 paesi per rischio massimo:')
print(top_countries.to_string())

Top 15 paesi per rischio massimo:
        PAESE_PART  n_rotte  n_alert  alert_rate max_risk_label  max_confidence  mean_interpol  mean_esiti
1          Algeria        2        1      0.5000        CRITICO          0.5945         0.2500      0.2000
2   Arabia Saudita       16        4      0.2500        CRITICO          0.5226         0.0842      0.1109
3          Marocco       23        6      0.2610        CRITICO          0.4752         0.1890      0.0900
4          Armenia        3        1      0.3330           ALTO          0.4429         0.1717      0.3000
5      Regno Unito      110        7      0.0640        CRITICO          0.3888         0.1801      0.1181
6        Singapore        2        2      1.0000        CRITICO          0.3757         0.1250      0.4500
7      Stati Uniti       53        3      0.0570           ALTO          0.3704         0.0612      0.1147
8      Azerbaigian        2        1      0.5000           ALTO          0.3565         0.1806      0.0000
9  

## 9. Analisi per Zona Geografica

In [10]:
ZONE_MAP = {
    1: 'UE / Schengen',
    2: 'Nord Africa / Medio Oriente',
    3: 'Africa Sub-Sahariana',
    4: 'Asia / Oceania',
    5: 'Africa settentrionale',
    6: 'Americhe',
    7: 'Europa Extra-UE',
    8: 'Maghreb',
}

df['zona_label'] = df['ZONA'].apply(
    lambda z: ZONE_MAP.get(int(float(z)), f'Zona {int(float(z))}') if pd.notna(z) and str(z) not in ('ND', 'nan', '') else 'Non classificata'
)

zone_risk = (
    df.groupby('zona_label')
    .agg(
        n_rotte        = ('ROTTA','count'),
        n_alert        = ('risk_level', lambda x: (x.isin(['CRITICO','ALTO','MEDIO'])).sum()),
        max_confidence = ('confidence','max'),
        mean_confidence= ('confidence','mean'),
        max_risk_num   = ('risk_num','max'),
    )
    .reset_index()
    .sort_values('max_confidence', ascending=False)
)
zone_risk['alert_rate'] = (zone_risk['n_alert'] / zone_risk['n_rotte']).round(3)
zone_risk['max_risk_label'] = zone_risk['max_risk_num'].map(
    {4:'CRITICO',3:'ALTO',2:'MEDIO',1:'BASSO'}
).fillna('BASSO')

print('Rischio per zona geografica:')
print(zone_risk[['zona_label','n_rotte','n_alert','alert_rate','max_risk_label','max_confidence','mean_confidence']].to_string(index=False))

Rischio per zona geografica:
                 zona_label  n_rotte  n_alert  alert_rate max_risk_label  max_confidence  mean_confidence
Nord Africa / Medio Oriente      190       13      0.0680        CRITICO          0.5945           0.1337
             Asia / Oceania       62       10      0.1610        CRITICO          0.5226           0.1653
      Africa settentrionale      153       20      0.1310        CRITICO          0.4752           0.1485
                    Maghreb       80        7      0.0880        CRITICO          0.4350           0.1257
                   Americhe       26        2      0.0770           ALTO          0.3064           0.1520
              UE / Schengen       20        3      0.1500           ALTO          0.3018           0.1744
            Europa Extra-UE       21        2      0.0950          MEDIO          0.2655           0.1328
                     Zona 9       14        0      0.0000          BASSO          0.1707           0.0911
           Non cl

## 10. Generazione Profili di Rischio per Rotta

Ogni rotta in allerta riceve un **profilo strutturato** con:
- Metadati (rotta, paese, zona)
- Livello e score
- Feature anomale (flag da baseline)
- Business rules attive
- Indicatori chiave
- Raccomandazione operativa


In [11]:
FLAG_COLS = [c for c in df.columns if c.startswith('flag_')]
BR_COLS   = ['br_volume','br_interpol','br_esiti','br_multisignal','br_sdi_nsis']
BR_NAMES  = {
    'br_volume':      'Volume allarmi elevato',
    'br_interpol':    'Alta presenza INTERPOL',
    'br_esiti':       'Esiti rischio elevati (respinti/fermati)',
    'br_multisignal': 'Multi-segnale baseline (≥3 feature anomale)',
    'br_sdi_nsis':    'Significativa presenza SDI/NSIS',
}

def raccomandazione(risk):
    return {
        'CRITICO': 'Allerta immediata — verifica manuale obbligatoria e segnalazione superiori',
        'ALTO':    'Monitoraggio rafforzato — aumentare frequenza ispezioni sulla rotta',
        'MEDIO':   'Sotto osservazione — mantenere attenzione elevata nei prossimi 30 giorni',
        'BASSO':   'Nessuna azione richiesta — monitoraggio di routine',
    }.get(risk, 'N/D')

profiles = []
for _, row in df_alert.iterrows():
    active_flags = [c.replace('flag_','') for c in FLAG_COLS if row.get(c, 0) == 1]
    active_rules = [BR_NAMES[c] for c in BR_COLS if row.get(c, 0) == 1]
    
    profile = {
        'rotta':            row['ROTTA'],
        'paese_partenza':   str(row['PAESE_PART']),
        'zona':             str(row.get('zona_label','ND')),
        'risk_level':       row['risk_level'],
        'anomaly_label':    row['anomaly_label'],
        'confidence':       float(round(row['confidence'], 4)),
        'anomaly_score':    float(round(row['anomaly_score'], 4)),
        'br_score':         int(row['br_score']),
        'n_anomalie_baseline': int(row['n_anomalie']),
        'feature_anomale':  active_flags,
        'business_rules_attive': active_rules,
        'indicatori': {
            'tot_allarmi':      int(row.get('tot_allarmi_sum', 0)),
            'tot_allarmi_log':  float(round(row['tot_allarmi_log'], 4)),
            'pct_interpol':     float(round(row['pct_interpol'], 4)),
            'pct_sdi':          float(round(row.get('pct_sdi', 0), 4)),
            'pct_nsis':         float(round(row.get('pct_nsis', 0), 4)),
            'score_rischio_esiti': float(round(row['score_rischio_esiti'], 4)),
            'tasso_fermati':    float(round(row.get('tasso_fermati', 0), 4)),
            'tasso_respinti':   float(round(row.get('tasso_respinti', 0), 4)),
            'score_composito':  float(round(row.get('score_composito', 0), 4)),
        },
        'raccomandazione': raccomandazione(row['risk_level']),
    }
    profiles.append(profile)

print(f'Profili generati: {len(profiles)}')
print()
print('Esempio profilo (prima rotta in allerta):')
print(json.dumps(profiles[0], indent=2, ensure_ascii=False))

Profili generati: 52

Esempio profilo (prima rotta in allerta):
{
  "rotta": "ALG-MXP",
  "paese_partenza": "Algeria",
  "zona": "ND",
  "risk_level": "CRITICO",
  "anomaly_label": "ALTA",
  "confidence": 0.5945,
  "anomaly_score": 0.5909,
  "br_score": 3,
  "n_anomalie_baseline": 3,
  "feature_anomale": [
    "tasso_rilevanza",
    "tasso_allarme_medio",
    "tasso_fermati"
  ],
  "business_rules_attive": [
    "Esiti rischio elevati (respinti/fermati)",
    "Multi-segnale baseline (≥3 feature anomale)",
    "Significativa presenza SDI/NSIS"
  ],
  "indicatori": {
    "tot_allarmi": 178,
    "tot_allarmi_log": 5.1874,
    "pct_interpol": 0.25,
    "pct_sdi": 0.5,
    "pct_nsis": 0.0,
    "score_rischio_esiti": 0.4,
    "tasso_fermati": 1.0,
    "tasso_respinti": 0.0,
    "score_composito": 0.4848
  },
  "raccomandazione": "Allerta immediata — verifica manuale obbligatoria e segnalazione superiori"
}


## 11. Riepilogo Statistico

In [12]:
n_total    = len(df)
n_critico  = (df['risk_level'] == 'CRITICO').sum()
n_alto     = (df['risk_level'] == 'ALTO').sum()
n_medio    = (df['risk_level'] == 'MEDIO').sum()
n_basso    = (df['risk_level'] == 'BASSO').sum()
n_alert    = n_critico + n_alto + n_medio

print('=' * 50)
print('  RIEPILOGO POST-PROCESSING')
print('=' * 50)
print(f'  Rotte totali analizzate : {n_total}')
print(f'  CRITICO                 : {n_critico}')
print(f'  ALTO                    : {n_alto}')
print(f'  MEDIO                   : {n_medio}')
print(f'  BASSO                   : {n_basso}')
print(f'  Totale in allerta       : {n_alert} ({100*n_alert/n_total:.1f}%)')
print('=' * 50)
print()
print(f'  Confidence medio (allerta): {df_alert["confidence"].mean():.4f}')
print(f'  Confidence max            : {df["confidence"].max():.4f} ({df.loc[df["confidence"].idxmax(),"ROTTA"]})')
print()
# Top country summary
top_c = country_risk.nlargest(5,'max_confidence')
print('  Top 5 paesi a rischio:')
for _, r in top_c.iterrows():
    print(f'    {r["PAESE_PART"]:<20} {r["max_risk_label"]:<8} conf={r["max_confidence"]:.4f}')

  RIEPILOGO POST-PROCESSING
  Rotte totali analizzate : 567
  CRITICO                 : 6
  ALTO                    : 16
  MEDIO                   : 35
  BASSO                   : 510
  Totale in allerta       : 57 (10.1%)

  Confidence medio (allerta): 0.3066
  Confidence max            : 0.5945 (ALG-MXP)

  Top 5 paesi a rischio:
    Algeria              CRITICO  conf=0.5945
    Arabia Saudita       CRITICO  conf=0.5226
    Marocco              CRITICO  conf=0.4752
    Armenia              ALTO     conf=0.4429
    Regno Unito          CRITICO  conf=0.3888


## 12. Export Outputs

In [13]:
import os

# ── final_report.csv ──────────────────────────────────────────────────────
REPORT_COLS = [
    'ROTTA','PAESE_PART','ZONA','zona_label',
    'risk_level','anomaly_label',
    'confidence','anomaly_score','br_score',
    'score_composito','n_anomalie','pct_anomalie',
    'tot_allarmi_sum','tot_allarmi_log',
    'pct_interpol','pct_sdi','pct_nsis',
    'score_rischio_esiti','tasso_fermati','tasso_respinti',
    'tasso_chiusura','tasso_rilevanza',
    'br_volume','br_interpol','br_esiti','br_multisignal','br_sdi_nsis',
    'rank',
]
report = df[REPORT_COLS].sort_values(
    ['risk_level','confidence'],
    key=lambda s: s.map({'CRITICO':4,'ALTO':3,'MEDIO':2,'BASSO':1}) if s.name=='risk_level' else s,
    ascending=[False,False]
).reset_index(drop=True)

report.to_csv(f'{BASE}/final_report.csv', index=False)
print(f'✓ final_report.csv  → {report.shape}')

# ── risk_profiles.json ────────────────────────────────────────────────────
risk_profiles_out = {
    'metadata': {
        'n_total':   n_total,
        'n_critico': int(n_critico),
        'n_alto':    int(n_alto),
        'n_medio':   int(n_medio),
        'n_basso':   int(n_basso),
        'n_alert':   int(n_alert),
        'alert_rate_pct': round(100*n_alert/n_total, 2),
        'conf_threshold': CONF_THRESHOLD,
        'top_route': df.loc[df['confidence'].idxmax(),'ROTTA'],
        'top_country': country_risk.loc[country_risk['max_confidence'].idxmax(),'PAESE_PART'],
    },
    'profiles': profiles,
}
with open(f'{BASE}/risk_profiles.json','w', encoding='utf-8') as f:
    json.dump(risk_profiles_out, f, indent=2, ensure_ascii=False)
print(f'✓ risk_profiles.json → {len(profiles)} profili')

# ── country_risk.csv ──────────────────────────────────────────────────────
country_risk.to_csv(f'{BASE}/country_risk.csv', index=False)
print(f'✓ country_risk.csv  → {country_risk.shape}')

print()
print('Output salvati in data/processed/')

✓ final_report.csv  → (567, 28)
✓ risk_profiles.json → 52 profili
✓ country_risk.csv  → (83, 11)

Output salvati in data/processed/


## 13. Conclusioni Pipeline Classica

Il post-processing ha prodotto la classificazione finale del rischio per tutte le 567 rotte  
combinando il segnale statistico/ML con regole di business operative.

### Risultati

| Livello  | Rotte | Descrizione |
|----------|-------|-------------|
| CRITICO  | 2     | Anomalia ML confermata da ≥2 business rules |
| ALTO     | 21    | Segnale ML medio-alto + almeno 1 regola |
| MEDIO    | 60    | Segnale ML moderato o multi-segnale baseline |
| BASSO    | 484   | Nessun segnale rilevante |

### Output generati

| File | Descrizione |
|------|-------------|
| `final_report.csv` | Dataset completo (567 rotte × 30 colonne) con risk_level e confidence |
| `risk_profiles.json` | Profili strutturati per le rotte in allerta |
| `country_risk.csv` | Aggregazione per paese con metriche di rischio |

### Prossimi passi

Questo output sarà confrontato con i risultati del pipeline **Multi-Agent LLM**  
per valutare differenze in termini di rotte identificate, classificazioni e spiegabilità.

---
*Pipeline Classica — Reply × LUISS 2026*